<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_GeoPolitical_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

# Middle East Oil Security Simulation Engine
MOS is an LLM-driven geopolitical simulation in which strategic actors choose actions within an explicitly represented and evolving WorldState. Chanakya proposes actor-specific strategic options, Centaur evaluates their relative likelihood, and ZeitWorld translates the selected Event into the next WorldState. The objective is not to predict the future, but to explore plausible, path-dependent trajectories arising from alternative strategic choices.

In [1]:
!pip install -q python-docx
#from docx.oxml import OxmlElement
#from docx.oxml.ns import qn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 8.7 MB/s eta 0:00:00


In [2]:
!wget -O CentaurLogo.png -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/logo-S.png

!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py
!wget -O mosZeitWorld.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosZeitWorld.py
!wget -O mosChanakya.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosChanakya.py
!wget -O mosCentaur.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosCentaur.py
!wget -O mosReporter.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosReporter.py


from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *
from mosReporter import *

In [3]:
import importlib

import mos
import mosZeitWorld
import mosChanakya
import mosCentaur
import mosReporter

importlib.reload(mos)
importlib.reload(mosZeitWorld)
importlib.reload(mosChanakya)
importlib.reload(mosCentaur)
importlib.reload(mosReporter)

from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *
from mosReporter import *

## Initialise Runtime Environment

Authenticate OpenAI, select the working LLM, mount Google Drive and define the paths to the MOS data, roles and prompts.

The directory listings provide a quick check that all required resources are available.

In [4]:
client = authenticateOpenAI()
model = "gpt-5.6-luna"

# Real-world evidence, LLM roles and prompts
# are stored persistently in Google Drive.

from google.colab import drive
drive.mount('/content/drive')

MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
WORLDFACTS_DIR = f"{MOS_HOME}/WorldFacts"
ZW_PATH = f"{MOS_HOME}/ZeitWorld"
CH_PATH = f"{MOS_HOME}/Chanakya"
CT_PATH = f"{MOS_HOME}/Centaur"

#!ls -l "{WORLDFACTS_DIR}"
#!ls -lh "{WORLDFACTS_DIR}/MD"
#!ls -lh "{WORLDFACTS_DIR}/NEWS"
#!ls -lh "{ZW_PATH}"
#!ls -lh "{CH_PATH}"
#!ls -lh "{CT_PATH}"

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com
Mounted at /content/drive


## Get World Facts

In [5]:
# Build WorldFactBase
#
# Static PDF-derived Markdown files and live Moneycontrol news
# are assembled into a single corpus used to initialise W0.
# No real-world information enters the simulation after W0.
# ------------------------------------------------------------

WorldFactBase = getWorldFactBase(WORLDFACTS_DIR)

Static sources : 8
Live sources   : 1
Characters     : 235,072
Words          : 35,600


### Defining the World

In [6]:
# Define the WorldState variables
#
# Eight variables currently describe the simulated world.
# Additional variables may be added if required.
# All variables begin at index = 100 in W0.
# ------------------------------------------------------------

worldVariables = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction="Higher index means higher oil price."
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction="Higher index means greater oil flow."
    ),

    WorldVariable(
        name="India Gulf Oil Imports",
        description=(
            "Volume of crude oil imported by India from "
            "Gulf-origin oil-producing countries."
        ),
        direction=(
            "Higher index means greater Indian imports "
            "of Gulf-origin oil."
        )
    ),

    WorldVariable(
        name="Iran Sanctions Enforcement",
        description=(
            "Degree to which restrictions on Iranian petroleum exports, "
            "trade and financial transactions are effectively enforced "
            "rather than circumvented."
        ),
        direction=(
            "Higher index means more effective sanctions enforcement "
            "and less successful circumvention."
        )
    ),

    WorldVariable(
        name="US/Allied Cohesion",
        description=(
            "Degree of alignment between the United States and cooperating "
            "states in their response to Iran and the Gulf crisis."
        ),
        direction="Higher index means greater US/allied cohesion."
    ),

    WorldVariable(
        name="Iran Internal Cohesion",
        description=(
            "Degree of alignment within Iran's political, military and "
            "security establishment in responding to the crisis."
        ),
        direction="Higher index means greater internal Iranian cohesion."
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction="Higher index means greater US-Iran tension."
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )
]

## Agents and their Roles

### ZeitWorld

In [7]:
# Load ZeitWorld roles and prompts
#
# ZW_Role                  : defines ZeitWorld's role and discipline
# ZW_Initialisation_Prompt : creates W0 from the WorldFactBase
# ZW_Update_Prompt         : evolves the current WorldState after an Event
# ------------------------------------------------------------

ZW_Role = getPrompt(f"{ZW_PATH}/ZW_Role_v0.txt")
ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v1.txt")
ZW_Update_Prompt = getPrompt(f"{ZW_PATH}/ZW_Update_Prompt_v0.txt")

In [8]:
# Instantiate ZeitWorld
#
# ZeitWorld is created with its LLM client, model and governing role.
# The initialised flag confirms whether W0 has already been created.
# ------------------------------------------------------------

ZeitWorld = ZeitWorld_Agent(client, model, ZW_Role)

#----------
#print(ZeitWorld.initialised)
#print(ZeitWorld.model)

### Chanakya

In [9]:
# Load Chanakya roles and strategy
#
# Generic Role : defines the common strategic discipline of all Chanakyas
# US Role      : defines US objectives, interests and constraints
# Iran Role    : defines Iranian objectives, interests and constraints
# BinaryHD     : generates two contrasting Hawk and Dove candidates
# ============================================================

CH_Generic_Role = getPrompt(f"{CH_PATH}/CH_Role_Generic_v0.txt")
CH_US_Role      = getPrompt(f"{CH_PATH}/CH_Role_US_v0.txt")
CH_IR_Role      = getPrompt(f"{CH_PATH}/CH_Role_Iran_v0.txt")
CH_BinaryHD     = getPrompt(f"{CH_PATH}/CH_Strategy_BinaryHD_v0.txt")

In [10]:
# Instantiate Chanakya agents
#
# Each Chanakya represents a national actor with its own
# interests and constraints, using the current Binary Hawk/Dove strategy.
# The registry maps each actor to its corresponding Chanakya.
# ============================================================

Chanakya_US = Chanakya_Agent(
    agent_id="ChanakyaUS_HD",
    actor="US",
    generic_role=CH_Generic_Role,
    specific_role=CH_US_Role,
    strategy=CH_BinaryHD
)

Chanakya_US.show()

Chanakya_IR = Chanakya_Agent(
    agent_id="ChanakyaIR_HD",
    actor="Iran",
    generic_role=CH_Generic_Role,
    specific_role=CH_IR_Role,
    strategy=CH_BinaryHD
)

Chanakya_IR.show()

ChanakyaRegistry = {
    "US": Chanakya_US,
    "Iran": Chanakya_IR
}

Chanakya : ChanakyaUS_HD
Actor    : US
Chanakya : ChanakyaIR_HD
Actor    : Iran


### Centaur

In [11]:
# Load Centaur roles
#
# Generic Role : defines the common adjudication discipline of all Centaurs
# UNSG Role    : provides a fair, independent UN diplomatic perspective
# ============================================================

CT_Generic_Role = getPrompt(f"{CT_PATH}/CT_Generic_Role_v0.txt")
CT_UNSG_Role      = getPrompt(f"{CT_PATH}/CT_UNSG_Role_v0.txt")

In [12]:
# Instantiate Centaur adjudicator
#
# CentaurUNSG evaluates Chanakya's Candidate Events from the
# perspective of a fair and independent UN Secretary-General.
# ============================================================
Centaur_UNSG = Centaur_Agent(
    agent_id="CentaurUNSG",
    generic_role=CT_Generic_Role,
    adjudication_role=CT_UNSG_Role
)

## Initialisation of Scenario

In [13]:
# Initialise Simulation State
#
# W stores the evolving WorldStates of the simulation.
# E stores the sequence of Events defining the Scenario.
# W0_Template defines the empty structure from which W0 is created.
# ------------------------------------------------------------

E = []
W = []

W0_Template = WorldState(
    state_id="W0",
    variables=worldVariables
)

# Initialise the first WorldState
#
# ZeitWorld uses the WorldFactBase and W0 template to create W0.
# The completed WorldState is automatically appended to W as W[0].
# ============================================================

_ = ZeitWorld.initialise(W,W0_Template, WorldFactBase, ZW_Initialisation_Prompt)

#print(ZeitWorld.initialised)

Initialising WorldState ... 
Model: gpt-5.6-luna | Effort: medium | Input: 57,064 | Output: 1,959 | Reasoning: 158 | Total: 59,023


In [14]:
# A first look at the intial World State W[0]
# The initial values are all set to 100, as the baseline
#

#W[0].show(-2)  # ---- just the world vectors
#W[0].show()    # ---- the entire state
#W[0].show(4)    # ---- details of any one vector

In [15]:
# Define initial Trigger options
#
# Two alternative Events are available to start the simulation.
# One is manually selected as E0; thereafter Events are generated
# and selected by the simulation.
# ============================================================

Trigger1 = Event(
    actor="Iran",
    action=(
        "Iran declares the Strait of Hormuz closed to all commercial shipping, "
        "orders its forces to prevent tanker transit, and launches attacks against "
        "US naval forces operating in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


Trigger2 = Event(
    actor="US",
    action=(
        "The United States announces an immediate suspension of offensive military "
        "operations against Iran and offers direct negotiations on maritime security "
        "and sanctions relief, while maintaining defensive protection of US forces "
        "and commercial shipping in the Gulf."
    ),
    rationale = "Initial Trigger for US"
)


In [16]:
# Select the initial Trigger
#
# The manually selected DefaultTrigger starts the first epoch.
# Its selection weight is None because it was not chosen by Shakuni.
# ============================================================
#DefaultTrigger = Trigger1
DefaultTrigger = Trigger2
DefaultTrigger.show()
print(DefaultTrigger.selection_weight)


Event     : 
Actor     : US
Action    : The United States announces an immediate suspension of offensive military operations against Iran and offers direct negotiations on maritime security and sanctions relief, while maintaining defensive protection of US forces and commercial shipping in the Gulf.
Rationale : Initial Trigger for US
selection_weight : None
None


In [17]:
# Initialise / Reset Scenario
#
# Retains W0 and clears all subsequent WorldStates and Events,
# restoring the simulation to its starting condition.
# ============================================================

def resetWorld():
    del W[1:]
    E.clear()

    print("Scenario reset to W0.")

# ------------------------------------------------------------

resetWorld()

Scenario reset to W0.


### Define Single Epoch

In [18]:
def runEpoch():
    # ============================================================
    # EPOCH STEP 1 — Establish Current Trigger
    #
    # The epoch is derived from the current WorldState history.
    # For the first epoch, DefaultTrigger becomes E0.
    # Thereafter, the last Reaction becomes the next Trigger.
    # ============================================================

    epoch = len(W) - 1
    if len(E) == 0:
        DefaultTrigger.event_id = f"E{epoch}"
        E.append(DefaultTrigger)
    else:
        assert E[epoch] is E[-1]

    #E[epoch].show()

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 1 complete" )

    # ============================================================
    # EPOCH STEP 2 — Update WorldState
    #
    # ZeitWorld applies Trigger E[epoch] to W[epoch] and creates
    # the next WorldState W[epoch+1].
    # ============================================================

    _ = ZeitWorld.update( W,E[epoch],ZW_Update_Prompt)

    #E[epoch].show()
    #W[epoch+1].show(-2)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 2 complete" )

    # ============================================================
    # EPOCH STEP 3 — Select Active Actor
    #
    # The Turn Controller identifies which actor responds to the
    # Current Trigger and activates its corresponding Chanakya.
    # ============================================================

    #ActiveChanakya = getActiveChanakya(E[-1], ChanakyaRegistry)
    ActiveChanakya = getActiveChanakya(E[epoch], ChanakyaRegistry)
    #ActiveChanakya.show()

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 3 complete" )

    # ============================================================
    # EPOCH STEP 4 — Generate Candidate Reactions
    #
    # The Active Chanakya evaluates the updated WorldState and
    # generates Candidate Events as possible reactions to the Trigger.
    # ============================================================

    Candidates = ActiveChanakya.propose(
        client=client,
        model=model,
        trigger=E[epoch],
        world=W[epoch+1]
    )

    #for i, candidate in enumerate(Candidates):
    #    print(f"\nCandidate Reaction {i+1}")
    #    print("-" * 60)
    #    candidate.show()

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 4 complete" )

    # ============================================================
    # EPOCH STEP 5 — Adjudicate Candidate Reactions
    #
    # Centaur evaluates the Candidate Events against the current
    # WorldState and assigns each a relative simulation weight.
    # ============================================================

    weights = Centaur_UNSG.evaluate(
        client,
        model,
        E[epoch],
        W[epoch + 1],
        Candidates
    )

    #print("\nCentaur Weights")
    #print("----------------")
    print("Weights",weights)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 5 complete" )

    # ============================================================
    #
    # Centaur weights represent the relative conditional likelihood
    # of each Candidate at this decision point.
    #
    # Shakuni makes a single stochastic selection using these weights.
    # The selected Candidate becomes the Reaction, retains its Centaur
    # weight as selection_weight, receives the next Event ID, and is
    # appended to E as the Trigger for the next epoch.
    # ============================================================

    Reaction = Shakuni(Candidates, weights)

    Reaction.event_id = f"E{len(E)}"
    E.append(Reaction)

    #print("\nShakuni has spoken")
    #print("------------------")
    #Reaction.show()
    #print(Reaction.selection_weight)

    print("epoch :", epoch, "len(W):", len(W), "len(E):", len(E), " ***** Step # 6 (last) complete" )
    print("========================================================================================")



##Scenario | Multiple Epochs

In [19]:
MaxEvents = 3

resetWorld()

while len(E) < MaxEvents:
    runEpoch()

# Apply the final Event and close the Scenario
_ = ZeitWorld.update(W, E[-1], ZW_Update_Prompt)

# Structural integrity check
assert len(E) == MaxEvents
assert len(W) == MaxEvents + 1
assert len(W) == len(E) + 1

print("\nScenario complete")
print("-----------------")
print("Events      :", len(E))
print("WorldStates :", len(W))

Scenario reset to W0.
epoch : 0 len(W): 1 len(E): 1  ***** Step # 1 complete
Updating WorldState ---------- W0
Model: gpt-5.6-luna | Effort: medium | Input: 3,130 | Output: 2,231 | Reasoning: 269 | Total: 5,361
Appended  WorldState ---------- W1
epoch : 0 len(W): 2 len(E): 1  ***** Step # 2 complete
epoch : 0 len(W): 2 len(E): 1  ***** Step # 3 complete
Chanakya working ---------- ChanakyaIR_HD
Model: gpt-5.6-luna | Effort: medium | Input: 2,255 | Output: 298 | Reasoning: 65 | Total: 2,553
epoch : 0 len(W): 2 len(E): 1  ***** Step # 4 complete
Centaur working ---------- CentaurUNSG
Model: gpt-5.6-luna | Effort: medium | Input: 2,127 | Output: 65 | Reasoning: 49 | Total: 2,192
Weights [0.62, 0.38]
epoch : 0 len(W): 2 len(E): 1  ***** Step # 5 complete
epoch : 0 len(W): 2 len(E): 2  ***** Step # 6 (last) complete
epoch : 1 len(W): 2 len(E): 2  ***** Step # 1 complete
Updating WorldState ---------- W1
Model: gpt-5.6-luna | Effort: medium | Input: 3,286 | Output: 2,410 | Reasoning: 228 | T

## Reporter

In [20]:
MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
#WORLDFACTS_DIR = f"{MOS_HOME}/WorldFacts"
#ZW_PATH = f"{MOS_HOME}/ZeitWorld"
#CH_PATH = f"{MOS_HOME}/Chanakya"
#CT_PATH = f"{MOS_HOME}/Centaur"
PR_PATH = f"{MOS_HOME}/Reporter"

#!ls -l "{WORLDFACTS_DIR}"
#!ls -lh "{WORLDFACTS_DIR}/MD"
#!ls -lh "{WORLDFACTS_DIR}/NEWS"
#!ls -lh "{ZW_PATH}"
#!ls -lh "{CH_PATH}"
#!ls -lh "{CT_PATH}"
#!ls -lh "{PR_PATH}"

In [21]:
# ============================================================
# Load PressReporter Roles and Prompts
#
# Generic Role defines reporting discipline.
# Narrative Prompt produces the human-readable Scenario history.
# Structured Prompt produces the machine-oriented Markdown record.
# ============================================================

PR_Generic_Role = getPrompt(f"{PR_PATH}/PR_Generic_Role_v0.txt")
PR_Narrative    = getPrompt(f"{PR_PATH}/PR_Narrative_v0.txt")
PR_Structured   = getPrompt(f"{PR_PATH}/PR_Structured_v0.txt")


# ============================================================
# Instantiate PressReporter
# ============================================================

PressReporter = PressReporter_Agent(
    client=client,
    model=model,
    agent_id="PressReporter",
    generic_role=PR_Generic_Role
)

In [22]:
#ScenarioText = PressReporter._scenarioText(W, E)
#print(ScenarioText)

In [23]:
# ============================================================
# Generate Narrative Scenario Report
#
# PressReporter converts the completed Scenario (W + E) into a
# human-readable narrative using the Narrative reporting prompt.
# The resulting report is then formatted and saved as a Word document.
# ============================================================
NarrativeReport = PressReporter.narrative(
    W,
    E,
    PR_Narrative
)

#print(NarrativeReport)

NarrativeFile = createNarrativeDocx(
    NarrativeReport,
    E,
    scenarioID="S001",
    title="Middle East Oil Security — Scenario Report"
)

PressReporter working ---------- PressReporter | Narrative
Model: gpt-5.6-luna | Effort: medium | Input: 8,300 | Output: 1,068 | Reasoning: 115 | Total: 9,368
Word report created: /content/MOS_S001_261003_114839.docx


In [24]:
# ============================================================
# Generate Structured Scenario Report — Deferred
#
# Produces a structured Markdown representation of the completed
# Scenario for subsequent machine-based analysis.
# Currently disabled; retained for future use.
# ============================================================
'''
StructuredReport = PressReporter.structured(
    W,
    E,
    PR_Structured
)

#print(StructuredReport)
'''

'\nStructuredReport = PressReporter.structured(\n    W,\n    E,\n    PR_Structured\n)\n\n#print(StructuredReport)\n'

#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)